# Whoop Drone – PPO Training Pipeline (Colab / Kaggle)

**Stage 1** – Base hover policy (clean simulation, 5 M steps)

**Stage 2** – Domain-randomization fine-tuning (sim-to-real hardening, 2 M steps)

> Run cells top-to-bottom. All source files are embedded; no git clone needed.
> Artefacts are packaged and downloaded in the final cell.


In [ ]:
# ── 1. Install Python packages ─────────────────────────────────────────────
import subprocess, sys, os, warnings

# Silence the verbose CUDA / computation-placer warnings that Colab
# prints at import time (harmless duplicate-registration messages from
# pre-loaded TF/JAX).
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")   # PPO/MLP runs on CPU
warnings.filterwarnings("ignore")

# Remove the unmaintained legacy 'gym' package that ships with Colab/Kaggle
# base images. It intercepts 'import gym' and triggers deprecation noise.
subprocess.call(
    [sys.executable, "-m", "pip", "uninstall", "-q", "-y", "gym"],
    stderr=subprocess.DEVNULL,
)

_pkgs = [
    "mujoco>=3.1.0",
    "gymnasium>=0.29.0",
    "stable-baselines3>=2.3.0",
    "shimmy>=0.2.0",       # gym-gymnasium compatibility shim (SB3 needs it)
    "pyyaml",
    "rich",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _pkgs)
print("All packages installed.")


In [ ]:
# ── 2. Detect Colab / Kaggle, create directories ──────────────────────────
import os, sys

try:
    import google.colab   # noqa
    _ON_COLAB  = True
except ImportError:
    _ON_COLAB  = False

_ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if _ON_COLAB:
    BASE_DIR = "/content/whoop_drone"
elif _ON_KAGGLE:
    BASE_DIR = "/kaggle/working/whoop_drone"
else:
    BASE_DIR = os.path.join(os.getcwd(), "whoop_drone")

MODEL_DIR = os.path.join(BASE_DIR, "models")
ENV_DIR   = os.path.join(BASE_DIR, "envs")
LOG_DIR   = os.path.join(BASE_DIR, "logs")
SAVE_DIR  = os.path.join(BASE_DIR, "models", "trained", "best")

for _d in [MODEL_DIR, ENV_DIR, LOG_DIR, SAVE_DIR]:
    os.makedirs(_d, exist_ok=True)

# Add project root to Python path so 'import envs' works
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

print(f"Platform : {'Colab' if _ON_COLAB else ('Kaggle' if _ON_KAGGLE else 'Other')}")
print(f"BASE_DIR : {BASE_DIR}")


In [ ]:
# ── 3. Write whoop_drone.xml ────────────────────────────────────────────────
import base64, os

_XML_B64 = "PG11am9jbyBtb2RlbD0id2hvb3BfZHJvbmUiPgogIDwhLS0KICAgIDY1bW0gV2hvb3AgRHJvbmUgTXVKb0NvIE1vZGVsCiAgICA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PQogICAgTWFzcyAgICAgICA6IDUwZyB0b3RhbAogICAgRnJhbWUgICAgICA6IFggY29uZmlndXJhdGlvbiwgNjVtbSBtb3Rvci10by1tb3RvciBkaWFnb25hbAogICAgTW90b3JzICAgICA6IDR4IGJydXNobGVzcyAoMDgwMi8xMTAyIGNsYXNzKQogICAgTWF4IHRocnVzdCA6IH4wLjI0NSBOICh+MjVnKSBwZXIgbW90b3IgIOKGkiAgaG92ZXIgYXQgfjUwJSB0aHJvdHRsZQogICAgWWF3IHRvcnF1ZSA6IDAuMDEzIE7Ct20gcGVyIG1vdG9yIGF0IG1heCB0aHJvdHRsZSAoQ20vQ3Qg4omIIDAuMDUzKQoKICAgIE1vdG9yIGxheW91dCAodG9wIHZpZXcpOgogICAgICBGTCAoK3gsK3kpIENDVyAgIEZSICgreCwteSkgQ1cKICAgICAgICAgICAgXCAgICAgICAgICAvCiAgICAgICAgICAgICBcICAgICAgICAvCiAgICAgICAgICAgICAgLS0tLS0tLS0gIChjZW50ZXIpCiAgICAgICAgICAgICAvICAgICAgICBcCiAgICAgICAgICAgIC8gICAgICAgICAgXAogICAgICBCTCAoLXgsK3kpIENXICAgQlIgKC14LC15KSBDQ1cKCiAgICBDb29yZGluYXRlIGZyYW1lOiB4PWZvcndhcmQsIHk9bGVmdCwgej11cAogICAgQWN0aW9uIG9yZGVyOiBbRkwsIEZSLCBCTCwgQlJdIGVhY2ggaW4gWzAsIDFdCgogICAgPT09IFRVTklORyBHVUlERSA9PT0KICAgIElmIGRyb25lIG9zY2lsbGF0ZXMgIOKGkiByZWR1Y2UgdGhydXN0L3RvcnF1ZSBnZWFyIHZhbHVlcwogICAgSWYgZHJvbmUgaXMgc2x1Z2dpc2gg4oaSIGluY3JlYXNlIGdlYXIgdmFsdWVzIG9yIHJlZHVjZSBpbmVydGlhCiAgICBVcGRhdGUgJ2RpYWdpbmVydGlhJyBhZnRlciB3ZWlnaGluZyByZWFsIGRyb25lIGNvbXBvbmVudHMuCiAgICBVcGRhdGUgZ2VhciB2YWx1ZXMgYWZ0ZXIgbWVhc3VyaW5nIG1vdG9yIHRocnVzdCBvbiBhIHRocnVzdCBzdGFuZC4KICAtLT4KCiAgPGNvbXBpbGVyIGFuZ2xlPSJyYWRpYW4iIGF1dG9saW1pdHM9InRydWUiLz4KCiAgPG9wdGlvbiB0aW1lc3RlcD0iMC4wMDIiCiAgICAgICAgICBncmF2aXR5PSIwIDAgLTkuODEiCiAgICAgICAgICBpbnRlZ3JhdG9yPSJSSzQiCiAgICAgICAgICBjb25lPSJlbGxpcHRpYyIKICAgICAgICAgIGltcHJhdGlvPSIxMCIvPgoKICA8ZGVmYXVsdD4KICAgIDxnZW9tIGNvbnR5cGU9IjEiIGNvbmFmZmluaXR5PSIxIiBjb25kaW09IjMiCiAgICAgICAgICBmcmljdGlvbj0iMC43IDAuMDA1IDAuMDAwMSIgc29saW1wPSIwLjkgMC45NSAwLjAwMSIvPgogICAgPGpvaW50IGRhbXBpbmc9IjAuMDAwMSIvPgogIDwvZGVmYXVsdD4KCiAgPGFzc2V0PgogICAgPHRleHR1cmUgdHlwZT0ic2t5Ym94IiBidWlsdGluPSJncmFkaWVudCIKICAgICAgICAgICAgIHJnYjE9Ii40IC42IC44IiByZ2IyPSIwIDAgMCIKICAgICAgICAgICAgIHdpZHRoPSI1MTIiIGhlaWdodD0iNTEyIi8+CiAgICA8dGV4dHVyZSBuYW1lPSJncmlkIiB0eXBlPSIyZCIgYnVpbHRpbj0iY2hlY2tlciIKICAgICAgICAgICAgIHJnYjE9Ii4xNSAuMjUgLjM1IiByZ2IyPSIuMjUgLjM1IC40NSIKICAgICAgICAgICAgIHdpZHRoPSI1MTIiIGhlaWdodD0iNTEyIiBtYXJrPSJlZGdlIiBtYXJrcmdiPSIuMyAuNCAuNSIvPgogICAgPG1hdGVyaWFsIG5hbWU9ImdyaWQiIHRleHR1cmU9ImdyaWQiCiAgICAgICAgICAgICAgdGV4cmVwZWF0PSI0IDQiIHRleHVuaWZvcm09InRydWUiIHJlZmxlY3RhbmNlPSIwLjEiLz4KICAgIDxtYXRlcmlhbCBuYW1lPSJmcmFtZV9tYXQiICByZ2JhPSIwLjEyIDAuMTIgMC43NSAxLjAiLz4KICAgIDxtYXRlcmlhbCBuYW1lPSJmY19tYXQiICAgICByZ2JhPSIwLjE1IDAuNzUgMC4xNSAxLjAiLz4KICAgIDxtYXRlcmlhbCBuYW1lPSJtb3Rvcl9jY3ciICByZ2JhPSIwLjEgMC4xIDAuMSAxLjAiLz4KICAgIDxtYXRlcmlhbCBuYW1lPSJtb3Rvcl9jdyIgICByZ2JhPSIwLjc1IDAuMSAwLjEgMS4wIi8+CiAgICA8bWF0ZXJpYWwgbmFtZT0icHJvcF9tYXQiICAgcmdiYT0iMC40IDAuNCAwLjQgMC42Ii8+CiAgPC9hc3NldD4KCiAgPHdvcmxkYm9keT4KICAgIDxsaWdodCBuYW1lPSJzdW4iICAgIHBvcz0iMCAwIDUiICAgZGlyPSIwIDAgLTEiICBkaWZmdXNlPSIwLjggMC44IDAuOCIgc3BlY3VsYXI9IjAuMiAwLjIgMC4yIi8+CiAgICA8bGlnaHQgbmFtZT0iZmlsbCIgICBwb3M9IjMgLTMgNCIgIGRpcj0iLTEgMSAtMiIgZGlmZnVzZT0iMC4zIDAuMyAwLjMiLz4KCiAgICA8IS0tIEdyb3VuZCBwbGFuZSAtLT4KICAgIDxnZW9tIG5hbWU9ImZsb29yIiB0eXBlPSJwbGFuZSIgc2l6ZT0iMjUgMjUgMC4xIgogICAgICAgICAgbWF0ZXJpYWw9ImdyaWQiIHpheGlzPSIwIDAgMSIgY29udHlwZT0iMSIgY29uYWZmaW5pdHk9IjEiLz4KCiAgICA8IS0tID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KICAgICAgICAgRHJvbmUgYm9keSAg4oCTICBmcmVlam9pbnQgc28gaXQgY2FuIG1vdmUgZnJlZWx5IGluIHNwYWNlCiAgICAgICAgIEluaXRpYWwgcG9zaXRpb246IDEgbSBhYm92ZSBncm91bmQKICAgICAgICAgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PSAtLT4KICAgIDxib2R5IG5hbWU9ImRyb25lIiBwb3M9IjAgMCAxLjAiPgogICAgICA8ZnJlZWpvaW50IG5hbWU9ImRyb25lX2ZyZWVqb2ludCIvPgoKICAgICAgPCEtLQogICAgICAgIEluZXJ0aWFsIHByb3BlcnRpZXMgKDUwZyB3aG9vcCBlc3RpbWF0ZSkKICAgICAgICBJX3h4ID0gSV95eSDiiYggMi41ZS01IGtnwrdtwrIgIChyb2xsL3BpdGNoIOKAkyB0aGluIGRpc2sgZm9ybXVsYSkKICAgICAgICBJX3p6ICAgICAgICAg4omIIDQuMGUtNSBrZ8K3bcKyICAoeWF3KQogICAgICAgIE1lYXN1cmUgd2l0aCBhIGJpZmlsYXIgcGVuZHVsdW0gZm9yIGFjY3VyYXRlIHZhbHVlcy4KICAgICAgLS0+CiAgICAgIDxpbmVydGlhbCBtYXNzPSIwLjA1MCIKICAgICAgICAgICAgICAgIHBvcz0iMCAwIDAiCiAgICAgICAgICAgICAgICBkaWFnaW5lcnRpYT0iMi41ZS01IDIuNWUtNSA0LjBlLTUiLz4KCiAgICAgIDwhLS0gLS0tLSBGcmFtZSBhcm1zIChYIGNvbmZpZ3VyYXRpb24sIMKxNDXCsCkgLS0tLSAtLT4KICAgICAgPCEtLSBhcm0gZnJvbSBCTCgtLCspIHRvIEZSKCssLSkgZGlyZWN0aW9uIC0tPgogICAgICA8Z2VvbSBuYW1lPSJhcm1fMSIgdHlwZT0iYm94IgogICAgICAgICAgICBzaXplPSIwLjAyMzAgMC4wMDM1IDAuMDAyNSIKICAgICAgICAgICAgZXVsZXI9IjAgMCAgMC43ODU0IgogICAgICAgICAgICBtYXRlcmlhbD0iZnJhbWVfbWF0IiBjb250eXBlPSIwIiBjb25hZmZpbml0eT0iMCIvPgogICAgICA8IS0tIGFybSBmcm9tIEZMKCssKykgdG8gQlIoLSwtKSBkaXJlY3Rpb24gLS0+CiAgICAgIDxnZW9tIG5hbWU9ImFybV8yIiB0eXBlPSJib3giCiAgICAgICAgICAgIHNpemU9IjAuMDIzMCAwLjAwMzUgMC4wMDI1IgogICAgICAgICAgICBldWxlcj0iMCAwIC0wLjc4NTQiCiAgICAgICAgICAgIG1hdGVyaWFsPSJmcmFtZV9tYXQiIGNvbnR5cGU9IjAiIGNvbmFmZmluaXR5PSIwIi8+CgogICAgICA8IS0tIENlbnRyYWwgRkMvRVNDIHN0YWNrICh2aXN1YWwpIC0tPgogICAgICA8Z2VvbSBuYW1lPSJmYyIgdHlwZT0iYm94IgogICAgICAgICAgICBzaXplPSIwLjAxNCAwLjAxNCAwLjAwNSIKICAgICAgICAgICAgcG9zPSIwIDAgMC4wMDMiCiAgICAgICAgICAgIG1hdGVyaWFsPSJmY19tYXQiIGNvbnR5cGU9IjAiIGNvbmFmZmluaXR5PSIwIi8+CgogICAgICA8IS0tID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQogICAgICAgICAgIE1vdG9yIEZMICAoK3gsICt5KSAgQ0NXICDigJMgYmxhY2sgY2FwcwogICAgICAgICAgID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PSAtLT4KICAgICAgPGdlb20gbmFtZT0ibW90b3JfZmwiIHR5cGU9ImN5bGluZGVyIgogICAgICAgICAgICBzaXplPSIwLjAwNTUgMC4wMDYiCiAgICAgICAgICAgIHBvcz0iMC4wMzI1IDAuMDMyNSAwLjAwMyIKICAgICAgICAgICAgbWF0ZXJpYWw9Im1vdG9yX2NjdyIvPgogICAgICA8IS0tIFByb3BlbGxlciBGTCAodmlzdWFsLCBubyBjb2xsaXNpb24pIC0tPgogICAgICA8Z2VvbSBuYW1lPSJwcm9wX2ZsIiB0eXBlPSJjeWxpbmRlciIKICAgICAgICAgICAgc2l6ZT0iMC4wMzggMC4wMDA4IgogICAgICAgICAgICBwb3M9IjAuMDMyNSAwLjAzMjUgMC4wMTAiCiAgICAgICAgICAgIG1hdGVyaWFsPSJwcm9wX21hdCIKICAgICAgICAgICAgY29udHlwZT0iMCIgY29uYWZmaW5pdHk9IjAiLz4KICAgICAgPCEtLSBUaHJ1c3QgYXBwbGljYXRpb24gc2l0ZSBmb3IgRkwgLS0+CiAgICAgIDxzaXRlIG5hbWU9InNpdGVfZmwiIHBvcz0iMC4wMzI1IDAuMDMyNSAwLjAxMCIKICAgICAgICAgICAgemF4aXM9IjAgMCAxIiBzaXplPSIwLjAwMiIvPgoKICAgICAgPCEtLSA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KICAgICAgICAgICBNb3RvciBGUiAgKCt4LCAteSkgIENXICAg4oCTIHJlZCBjYXBzCiAgICAgICAgICAgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09IC0tPgogICAgICA8Z2VvbSBuYW1lPSJtb3Rvcl9mciIgdHlwZT0iY3lsaW5kZXIiCiAgICAgICAgICAgIHNpemU9IjAuMDA1NSAwLjAwNiIKICAgICAgICAgICAgcG9zPSIwLjAzMjUgLTAuMDMyNSAwLjAwMyIKICAgICAgICAgICAgbWF0ZXJpYWw9Im1vdG9yX2N3Ii8+CiAgICAgIDxnZW9tIG5hbWU9InByb3BfZnIiIHR5cGU9ImN5bGluZGVyIgogICAgICAgICAgICBzaXplPSIwLjAzOCAwLjAwMDgiCiAgICAgICAgICAgIHBvcz0iMC4wMzI1IC0wLjAzMjUgMC4wMTAiCiAgICAgICAgICAgIG1hdGVyaWFsPSJwcm9wX21hdCIKICAgICAgICAgICAgY29udHlwZT0iMCIgY29uYWZmaW5pdHk9IjAiLz4KICAgICAgPHNpdGUgbmFtZT0ic2l0ZV9mciIgcG9zPSIwLjAzMjUgLTAuMDMyNSAwLjAxMCIKICAgICAgICAgICAgemF4aXM9IjAgMCAxIiBzaXplPSIwLjAwMiIvPgoKICAgICAgPCEtLSA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KICAgICAgICAgICBNb3RvciBCTCAgKC14LCAreSkgIENXICAg4oCTIHJlZCBjYXBzCiAgICAgICAgICAgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09IC0tPgogICAgICA8Z2VvbSBuYW1lPSJtb3Rvcl9ibCIgdHlwZT0iY3lsaW5kZXIiCiAgICAgICAgICAgIHNpemU9IjAuMDA1NSAwLjAwNiIKICAgICAgICAgICAgcG9zPSItMC4wMzI1IDAuMDMyNSAwLjAwMyIKICAgICAgICAgICAgbWF0ZXJpYWw9Im1vdG9yX2N3Ii8+CiAgICAgIDxnZW9tIG5hbWU9InByb3BfYmwiIHR5cGU9ImN5bGluZGVyIgogICAgICAgICAgICBzaXplPSIwLjAzOCAwLjAwMDgiCiAgICAgICAgICAgIHBvcz0iLTAuMDMyNSAwLjAzMjUgMC4wMTAiCiAgICAgICAgICAgIG1hdGVyaWFsPSJwcm9wX21hdCIKICAgICAgICAgICAgY29udHlwZT0iMCIgY29uYWZmaW5pdHk9IjAiLz4KICAgICAgPHNpdGUgbmFtZT0ic2l0ZV9ibCIgcG9zPSItMC4wMzI1IDAuMDMyNSAwLjAxMCIKICAgICAgICAgICAgemF4aXM9IjAgMCAxIiBzaXplPSIwLjAwMiIvPgoKICAgICAgPCEtLSA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KICAgICAgICAgICBNb3RvciBCUiAgKC14LCAteSkgIENDVyAg4oCTIGJsYWNrIGNhcHMKICAgICAgICAgICA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0gLS0+CiAgICAgIDxnZW9tIG5hbWU9Im1vdG9yX2JyIiB0eXBlPSJjeWxpbmRlciIKICAgICAgICAgICAgc2l6ZT0iMC4wMDU1IDAuMDA2IgogICAgICAgICAgICBwb3M9Ii0wLjAzMjUgLTAuMDMyNSAwLjAwMyIKICAgICAgICAgICAgbWF0ZXJpYWw9Im1vdG9yX2NjdyIvPgogICAgICA8Z2VvbSBuYW1lPSJwcm9wX2JyIiB0eXBlPSJjeWxpbmRlciIKICAgICAgICAgICAgc2l6ZT0iMC4wMzggMC4wMDA4IgogICAgICAgICAgICBwb3M9Ii0wLjAzMjUgLTAuMDMyNSAwLjAxMCIKICAgICAgICAgICAgbWF0ZXJpYWw9InByb3BfbWF0IgogICAgICAgICAgICBjb250eXBlPSIwIiBjb25hZmZpbml0eT0iMCIvPgogICAgICA8c2l0ZSBuYW1lPSJzaXRlX2JyIiBwb3M9Ii0wLjAzMjUgLTAuMDMyNSAwLjAxMCIKICAgICAgICAgICAgemF4aXM9IjAgMCAxIiBzaXplPSIwLjAwMiIvPgoKICAgICAgPCEtLSBJTVUgc2l0ZSBhdCBkcm9uZSBDb00gLS0+CiAgICAgIDxzaXRlIG5hbWU9ImltdSIgcG9zPSIwIDAgMCIgemF4aXM9IjAgMCAxIiBzaXplPSIwLjAwMSIvPgogICAgPC9ib2R5PgogIDwvd29ybGRib2R5PgoKICA8IS0tID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KICAgICAgIEFjdHVhdG9ycwogICAgICAgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQogICAgICAgZ2VhciA9IFtmeCwgZnksIGZ6LCB0eCwgdHksIHR6XSAgaW4gc2l0ZSBMT0NBTCBmcmFtZS4KICAgICAgIGNvbnRyb2wg4oiIIFswLCAxXSAg4oaSICBmb3JjZSA9IGdlYXIgKiBjb250cm9sCgogICAgICAgdGhydXN0X21heCAgPSAwLjI0NSBOICAgKOKJiDI1ZyBwZXIgbW90b3IpCiAgICAgICAgIOKAoiBob3ZlciA9IDAuMDUga2cgw5cgOS44MSBtL3PCsiAvIDQgPSAwLjEyMyBOICDihpIgNTAlIHRocm90dGxlIOKckwogICAgICAgICDigKIgVC9XIGF0IGZ1bGwgdGhyb3R0bGUg4omIIDIuMCAgKGdvb2Qgc3RhYmlsaXR5IG1hcmdpbikKCiAgICAgICB0b3JxdWVfbWF4ICA9IDAuMDEzIE7Ct20gKHlhdyByZWFjdGlvbiB0b3JxdWUsIENtL0N0IMOXIHRocnVzdF9tYXgpCiAgICAgICAgIOKAoiBDQ1cgbW90b3JzIChGTCwgQlIpIOKGkiBwb3NpdGl2ZSB0egogICAgICAgICDigKIgQ1cgIG1vdG9ycyAoRlIsIEJMKSDihpIgbmVnYXRpdmUgdHoKCiAgICAgICBUbyB1cGRhdGUgZm9yIHJlYWwgbW90b3JzLCBtZWFzdXJlIHRocnVzdCBvbiBhIHN0YW5kIGFuZCByZXBsYWNlCiAgICAgICAwLjI0NSB3aXRoIG1lYXN1cmVkIHZhbHVlLiAgS2VlcCB0b3JxdWVfbWF4IOKJiCAwLjA1MyDDlyB0aHJ1c3RfbWF4LgogIC0tPgogIDxhY3R1YXRvcj4KICAgIDwhLS0gRkw6IENDVyDihpIgK3R6IC0tPgogICAgPGdlbmVyYWwgbmFtZT0ibW90b3JfZmwiIHNpdGU9InNpdGVfZmwiCiAgICAgICAgICAgICBnZWFyPSIwIDAgMC4yNDUgMCAwICAwLjAxMyIKICAgICAgICAgICAgIGN0cmxsaW1pdGVkPSJ0cnVlIiBjdHJscmFuZ2U9IjAgMSIvPgoKICAgIDwhLS0gRlI6IENXICDihpIgLXR6IC0tPgogICAgPGdlbmVyYWwgbmFtZT0ibW90b3JfZnIiIHNpdGU9InNpdGVfZnIiCiAgICAgICAgICAgICBnZWFyPSIwIDAgMC4yNDUgMCAwIC0wLjAxMyIKICAgICAgICAgICAgIGN0cmxsaW1pdGVkPSJ0cnVlIiBjdHJscmFuZ2U9IjAgMSIvPgoKICAgIDwhLS0gQkw6IENXICDihpIgLXR6IC0tPgogICAgPGdlbmVyYWwgbmFtZT0ibW90b3JfYmwiIHNpdGU9InNpdGVfYmwiCiAgICAgICAgICAgICBnZWFyPSIwIDAgMC4yNDUgMCAwIC0wLjAxMyIKICAgICAgICAgICAgIGN0cmxsaW1pdGVkPSJ0cnVlIiBjdHJscmFuZ2U9IjAgMSIvPgoKICAgIDwhLS0gQlI6IENDVyDihpIgK3R6IC0tPgogICAgPGdlbmVyYWwgbmFtZT0ibW90b3JfYnIiIHNpdGU9InNpdGVfYnIiCiAgICAgICAgICAgICBnZWFyPSIwIDAgMC4yNDUgMCAwICAwLjAxMyIKICAgICAgICAgICAgIGN0cmxsaW1pdGVkPSJ0cnVlIiBjdHJscmFuZ2U9IjAgMSIvPgogIDwvYWN0dWF0b3I+CgogIDwhLS0gPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQogICAgICAgU2Vuc29ycyAobWlycm9ycyBHWS05MSBvdXRwdXRzKQogICAgICAgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PSAtLT4KICA8c2Vuc29yPgogICAgPCEtLSBBY2NlbGVyb21ldGVyICYgZ3lybyBhdCBJTVUgc2l0ZSAoYm9keSBmcmFtZSkgLS0+CiAgICA8YWNjZWxlcm9tZXRlciBuYW1lPSJpbXVfYWNjZWwiIHNpdGU9ImltdSIvPgogICAgPGd5cm8gICAgICAgICAgbmFtZT0iaW11X2d5cm8iICBzaXRlPSJpbXUiLz4KCiAgICA8IS0tIEdyb3VuZCB0cnV0aCBzdGF0ZSAoZm9yIFJMIG9ic2VydmF0aW9uKSAtLT4KICAgIDxmcmFtZXBvcyAgICBuYW1lPSJkcm9uZV9wb3MiICAgICBvYmp0eXBlPSJib2R5IiBvYmpuYW1lPSJkcm9uZSIvPgogICAgPGZyYW1lcXVhdCAgIG5hbWU9ImRyb25lX3F1YXQiICAgIG9ianR5cGU9ImJvZHkiIG9iam5hbWU9ImRyb25lIi8+CiAgICA8ZnJhbWVsaW52ZWwgbmFtZT0iZHJvbmVfbGludmVsIiAgb2JqdHlwZT0iYm9keSIgb2JqbmFtZT0iZHJvbmUiLz4KICAgIDxmcmFtZWFuZ3ZlbCBuYW1lPSJkcm9uZV9hbmd2ZWwiICBvYmp0eXBlPSJib2R5IiBvYmpuYW1lPSJkcm9uZSIvPgogIDwvc2Vuc29yPgoKPC9tdWpvY28+Cg=="

_xml_path = os.path.join(MODEL_DIR, "whoop_drone.xml")
with open(_xml_path, "wb") as _f:
    _f.write(base64.b64decode(_XML_B64))
print(f"XML written → {_xml_path}")


In [ ]:
# ── 4. Write envs/drone_env.py and envs/drone_env_dr.py ────────────────────
import base64, os

# drone_env.py  ─────────────────────────────────────────────────────────
_ENV_B64 = "IiIiCldob29wRHJvbmVFbnYg4oCTIE11Sm9DbyBHeW1uYXNpdW0gZW52aXJvbm1lbnQgZm9yIGEgNTAgZyB3aG9vcC1jbGFzcyBkcm9uZS4KCk9ic2VydmF0aW9uICgxNy1kaW0pOgogICAgWzA6M10gIHBvc2l0aW9uIGVycm9yICAgKHdvcmxkIGZyYW1lLCBtKSAgICAgIDogcG9zIC0gdGFyZ2V0X3BvcwogICAgWzM6Nl0gIGxpbmVhciB2ZWxvY2l0eSAgKHdvcmxkIGZyYW1lLCBtL3MpCiAgICBbNjoxMF0gb3JpZW50YXRpb24gcXVhdGVybmlvbiBbdywgeCwgeSwgel0gICAgKGJvZHkgZnJhbWUpCiAgICBbMTA6MTNdIGFuZ3VsYXIgdmVsb2NpdHkgKHdvcmxkIGZyYW1lLCByYWQvcykKICAgIFsxMzoxN10gcHJldmlvdXMgbW90b3IgYWN0aW9ucyAgICAgICAgICAgICAgICAg4oiIIFswLCAxXQoKQWN0aW9uICg0LWRpbSk6IG1vdG9yIHRocm90dGxlcyBbRkwsIEZSLCBCTCwgQlJdIOKIiCBbMCwgMV0KICAgIEZMID0gZnJvbnQtbGVmdCAgKENDVyksICBGUiA9IGZyb250LXJpZ2h0IChDVykKICAgIEJMID0gYmFjay1sZWZ0ICAgKENXKSwgICBCUiA9IGJhY2stcmlnaHQgIChDQ1cpCgpUYXNrOiBob3ZlciBhdCB0YXJnZXRfcG9zIChkZWZhdWx0IFswLCAwLCAxXSBtKQoiIiIKCmltcG9ydCBvcwpmcm9tIHR5cGluZyBpbXBvcnQgRGljdCwgT3B0aW9uYWwsIFR1cGxlCgppbXBvcnQgbXVqb2NvCmltcG9ydCBtdWpvY28udmlld2VyCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgZ3ltbmFzaXVtIGFzIGd5bQpmcm9tIGd5bW5hc2l1bSBpbXBvcnQgc3BhY2VzCgoKY2xhc3MgV2hvb3BEcm9uZUVudihneW0uRW52KToKICAgIG1ldGFkYXRhID0geyJyZW5kZXJfbW9kZXMiOiBbImh1bWFuIiwgInJnYl9hcnJheSJdLCAicmVuZGVyX2ZwcyI6IDUwfQoKICAgICMg4pSA4pSAIFBoeXNpY2FsIGNvbnN0YW50cyAoc3luYyB3aXRoIHdob29wX2Ryb25lLnhtbCkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBNQVNTICAgICAgICAgID0gMC4wNTAgICAjIGtnCiAgICBHUkFWSVRZICAgICAgID0gOS44MSAgICAjIG0vc8KyCiAgICBNQVhfVEhSVVNUICAgID0gMC4yNDUgICAjIE4gcGVyIG1vdG9yCiAgICBBUk0gICAgICAgICAgID0gMC4wMzI1ICAjIG0gKG1vdG9y4oaSY2VudHJlIGRpc3RhbmNlLCBlYWNoIGF4aXMpCgogICAgIyBUaHJvdHRsZSByZXF1aXJlZCB0byBob3ZlciAocGVyIG1vdG9yKQogICAgSE9WRVJfVEhST1RUTEU6IGZsb2F0ID0gKE1BU1MgKiBHUkFWSVRZKSAvICg0LjAgKiBNQVhfVEhSVVNUKSAgICMg4omIIDAuNTAKCiAgICAjIOKUgOKUgCBTaW11bGF0aW9uIHRpbWluZyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICMgTXVKb0NvIHRpbWVzdGVwIGlzIDAuMDAyIHMgKDUwMCBIeikuCiAgICAjIG5fc3Vic3RlcHMgPSAxMCAg4oaSICA1MCBIeiBjb250cm9sIGZyZXF1ZW5jeS4KICAgIE5fU1VCU1RFUFMgPSAxMAoKICAgICMg4pSA4pSAIFRlcm1pbmF0aW9uIHRocmVzaG9sZHMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBDUkFTSF9aICAgICAgID0gMC4wMyAgICAjIG0gICDigJMgYmVsb3cgdGhpcyBoZWlnaHQgPSBjcmFzaAogICAgRkxJUF9XICAgICAgICA9IDAuMzAgICAgIyBxdWF0IHcg4oCTIGJlbG93IHRoaXMg4omIIHRpbHQgPiAxMDfCsAogICAgTUFYX1JBTkdFX1hZICA9IDEwLjAgICAgIyBtICAg4oCTIG1heCBob3Jpem9udGFsIGRpc3RhbmNlIGZyb20gb3JpZ2luCiAgICBNQVhfWiAgICAgICAgID0gMjAuMCAgICAjIG0gICDigJMgbWF4IGFsdGl0dWRlCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgcmVuZGVyX21vZGU6IE9wdGlvbmFsW3N0cl0gPSBOb25lLAogICAgICAgIHRhcmdldF9wb3M6IE9wdGlvbmFsW25wLm5kYXJyYXldID0gTm9uZSwKICAgICAgICBtYXhfZXBpc29kZV9zdGVwczogaW50ID0gMTAwMCwKICAgICk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCgogICAgICAgICMg4pSA4pSAIE11Sm9DbyBtb2RlbCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBfaGVyZSA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKQogICAgICAgIG1vZGVsX3BhdGggPSBvcy5wYXRoLmpvaW4oX2hlcmUsICIuLiIsICJtb2RlbHMiLCAid2hvb3BfZHJvbmUueG1sIikKICAgICAgICBzZWxmLm1vZGVsID0gbXVqb2NvLk1qTW9kZWwuZnJvbV94bWxfcGF0aChvcy5wYXRoLm5vcm1wYXRoKG1vZGVsX3BhdGgpKQogICAgICAgIHNlbGYuZGF0YSAgPSBtdWpvY28uTWpEYXRhKHNlbGYubW9kZWwpCgogICAgICAgICMgQ2FjaGUgYm9keSAvIGpvaW50IGlkcyBmb3IgZmFzdCBsb29rdXAKICAgICAgICBzZWxmLl9kcm9uZV9ib2R5X2lkID0gbXVqb2NvLm1qX25hbWUyaWQoCiAgICAgICAgICAgIHNlbGYubW9kZWwsIG11am9jby5tanRPYmoubWpPQkpfQk9EWSwgImRyb25lIgogICAgICAgICkKCiAgICAgICAgIyDilIDilIAgVGFzayBwYXJhbWV0ZXJzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHNlbGYudGFyZ2V0X3BvcyA9ICgKICAgICAgICAgICAgbnAuYXJyYXkodGFyZ2V0X3BvcywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICAgICAgaWYgdGFyZ2V0X3BvcyBpcyBub3QgTm9uZQogICAgICAgICAgICBlbHNlIG5wLmFycmF5KFswLjAsIDAuMCwgMS4wXSkKICAgICAgICApCiAgICAgICAgc2VsZi5tYXhfZXBpc29kZV9zdGVwcyA9IG1heF9lcGlzb2RlX3N0ZXBzCgogICAgICAgICMg4pSA4pSAIFNwYWNlcyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICAjIE9ic2VydmF0aW9uOiBwb3NfZXJyKDMpICsgdmVsKDMpICsgcXVhdCg0KSArIGFuZ192ZWwoMykgKyBwcmV2X2FjdCg0KQogICAgICAgIG9ic19sb3cgID0gbnAuYXJyYXkoCiAgICAgICAgICAgIFstNSwgIC01LCAgLTMsICAgICAgICAgICMgcG9zIGVycm9yCiAgICAgICAgICAgICAtMTAsIC0xMCwgLTEwLCAgICAgICAgICMgbGluZWFyIHZlbG9jaXR5CiAgICAgICAgICAgICAtMSwgIC0xLCAgLTEsICAtMSwgICAgIyBxdWF0ZXJuaW9uCiAgICAgICAgICAgICAtNTAsIC01MCwgLTUwLCAgICAgICAgICMgYW5ndWxhciB2ZWxvY2l0eQogICAgICAgICAgICAgIDAsICAgMCwgICAwLCAgIDBdLCAgICAjIHByZXYgYWN0aW9uCiAgICAgICAgICAgIGR0eXBlPW5wLmZsb2F0MzIsCiAgICAgICAgKQogICAgICAgIG9ic19oaWdoID0gbnAuYXJyYXkoCiAgICAgICAgICAgIFsgNSwgICA1LCAgIDUsCiAgICAgICAgICAgICAgMTAsICAxMCwgIDEwLAogICAgICAgICAgICAgICAxLCAgIDEsICAgMSwgIDEsCiAgICAgICAgICAgICAgNTAsICA1MCwgIDUwLAogICAgICAgICAgICAgICAxLCAgIDEsICAgMSwgIDFdLAogICAgICAgICAgICBkdHlwZT1ucC5mbG9hdDMyLAogICAgICAgICkKICAgICAgICBzZWxmLm9ic2VydmF0aW9uX3NwYWNlID0gc3BhY2VzLkJveChvYnNfbG93LCBvYnNfaGlnaCwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBzZWxmLmFjdGlvbl9zcGFjZSAgICAgID0gc3BhY2VzLkJveCgKICAgICAgICAgICAgbG93PTAuMCwgaGlnaD0xLjAsIHNoYXBlPSg0LCksIGR0eXBlPW5wLmZsb2F0MzIKICAgICAgICApCgogICAgICAgICMg4pSA4pSAIEludGVybmFsIHN0YXRlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHNlbGYuX3ByZXZfYWN0aW9uICA9IG5wLm9uZXMoNCwgZHR5cGU9bnAuZmxvYXQzMikgKiBzZWxmLkhPVkVSX1RIUk9UVExFCiAgICAgICAgc2VsZi5fc3RlcF9jb3VudCAgID0gMAoKICAgICAgICAjIOKUgOKUgCBSZW5kZXJpbmcg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgc2VsZi5yZW5kZXJfbW9kZSA9IHJlbmRlcl9tb2RlCiAgICAgICAgc2VsZi5fdmlld2VyICAgICA9IE5vbmUKICAgICAgICBzZWxmLl9yZW5kZXJlciAgID0gTm9uZQoKICAgICMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCiAgICAjIEludGVybmFsIGhlbHBlcnMKICAgICMg4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQ4pWQCgogICAgZGVmIF9nZXRfcG9zKHNlbGYpICAgICAtPiBucC5uZGFycmF5OiByZXR1cm4gc2VsZi5kYXRhLnFwb3NbMDozXQogICAgZGVmIF9nZXRfcXVhdChzZWxmKSAgICAtPiBucC5uZGFycmF5OiByZXR1cm4gc2VsZi5kYXRhLnFwb3NbMzo3XSAgICMgW3cseCx5LHpdCiAgICBkZWYgX2dldF92ZWwoc2VsZikgICAgIC0+IG5wLm5kYXJyYXk6IHJldHVybiBzZWxmLmRhdGEucXZlbFswOjNdCiAgICBkZWYgX2dldF9hbmdfdmVsKHNlbGYpIC0+IG5wLm5kYXJyYXk6IHJldHVybiBzZWxmLmRhdGEucXZlbFszOjZdCgogICAgZGVmIF9nZXRfb2JzKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgcG9zX2VyciA9IChzZWxmLl9nZXRfcG9zKCkgLSBzZWxmLnRhcmdldF9wb3MpLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHZlbCAgICAgPSBzZWxmLl9nZXRfdmVsKCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgcXVhdCAgICA9IHNlbGYuX2dldF9xdWF0KCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgYW5nX3ZlbCA9IHNlbGYuX2dldF9hbmdfdmVsKCkuYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKFtwb3NfZXJyLCB2ZWwsIHF1YXQsIGFuZ192ZWwsIHNlbGYuX3ByZXZfYWN0aW9uXSkKCiAgICBkZWYgX2dldF9pbmZvKHNlbGYpIC0+IERpY3Q6CiAgICAgICAgcG9zID0gc2VsZi5fZ2V0X3BvcygpCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInBvc2l0aW9uIjogICAgICAgICAgcG9zLmNvcHkoKSwKICAgICAgICAgICAgImRpc3RhbmNlX3RvX3RhcmdldCI6IGZsb2F0KG5wLmxpbmFsZy5ub3JtKHBvcyAtIHNlbGYudGFyZ2V0X3BvcykpLAogICAgICAgICAgICAic3RlcF9jb3VudCI6ICAgICAgICAgc2VsZi5fc3RlcF9jb3VudCwKICAgICAgICB9CgogICAgIyDilIDilIAgVGVybWluYXRpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgogICAgZGVmIF9jaGVja190ZXJtaW5hdGlvbihzZWxmKSAtPiBUdXBsZVtib29sLCBib29sXToKICAgICAgICAiIiJSZXR1cm5zICh0ZXJtaW5hdGVkLCB0cnVuY2F0ZWQpLiIiIgogICAgICAgIHBvcyAgPSBzZWxmLl9nZXRfcG9zKCkKICAgICAgICBxdWF0ID0gc2VsZi5fZ2V0X3F1YXQoKQoKICAgICAgICAjIENyYXNoZWQgaW50byBncm91bmQKICAgICAgICBpZiBwb3NbMl0gPCBzZWxmLkNSQVNIX1o6CiAgICAgICAgICAgIHJldHVybiBUcnVlLCBGYWxzZQogICAgICAgICMgRmxpcHBlZCBvdmVyCiAgICAgICAgaWYgYWJzKHF1YXRbMF0pIDwgc2VsZi5GTElQX1c6CiAgICAgICAgICAgIHJldHVybiBUcnVlLCBGYWxzZQogICAgICAgICMgT3V0IG9mIGJvdW5kcyDigJMgaG9yaXpvbnRhbAogICAgICAgIGlmIG5wLmFueShucC5hYnMocG9zWzoyXSkgPiBzZWxmLk1BWF9SQU5HRV9YWSk6CiAgICAgICAgICAgIHJldHVybiBUcnVlLCBGYWxzZQogICAgICAgICMgT3V0IG9mIGJvdW5kcyDigJMgYWx0aXR1ZGUKICAgICAgICBpZiBwb3NbMl0gPiBzZWxmLk1BWF9aOgogICAgICAgICAgICByZXR1cm4gVHJ1ZSwgRmFsc2UKICAgICAgICAjIFRpbWUgbGltaXQKICAgICAgICBpZiBzZWxmLl9zdGVwX2NvdW50ID49IHNlbGYubWF4X2VwaXNvZGVfc3RlcHM6CiAgICAgICAgICAgIHJldHVybiBGYWxzZSwgVHJ1ZQoKICAgICAgICByZXR1cm4gRmFsc2UsIEZhbHNlCgogICAgIyDilIDilIAgUmV3YXJkIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKICAgIGRlZiBfY29tcHV0ZV9yZXdhcmQoc2VsZiwgYWN0aW9uOiBucC5uZGFycmF5KSAtPiBmbG9hdDoKICAgICAgICBwb3MgICAgID0gc2VsZi5fZ2V0X3BvcygpCiAgICAgICAgcXVhdCAgICA9IHNlbGYuX2dldF9xdWF0KCkKICAgICAgICB2ZWwgICAgID0gc2VsZi5fZ2V0X3ZlbCgpCiAgICAgICAgYW5nX3ZlbCA9IHNlbGYuX2dldF9hbmdfdmVsKCkKCiAgICAgICAgcG9zX2VyciA9IHBvcyAtIHNlbGYudGFyZ2V0X3BvcwogICAgICAgIGRpc3QgICAgPSBmbG9hdChucC5saW5hbGcubm9ybShwb3NfZXJyKSkKCiAgICAgICAgIyAtLS0gUG9zaXRpb24gcmV3YXJkOiBleHBvbmVudGlhbCBzbyBncmFkaWVudCBkb2Vzbid0IHZhbmlzaCBmYXIgYXdheQogICAgICAgIHJfcG9zID0gZmxvYXQobnAuZXhwKC0yLjAgKiBkaXN0KioyKSkgLSAxLjAgICAgICAgICAgICMg4oiIIFstMSwgMF0KCiAgICAgICAgIyAtLS0gVXByaWdodCBvcmllbnRhdGlvbiByZXdhcmQgKHc9MSDihpIgbGV2ZWwgZmxpZ2h0KQogICAgICAgICMgICAgIHJfb3JpZW50ID0gIDEgd2hlbiBwZXJmZWN0bHkgdXByaWdodCwgIC0xIHdoZW4gZnVsbHkgaW52ZXJ0ZWQKICAgICAgICB3ICAgICAgICAgID0gZmxvYXQocXVhdFswXSkKICAgICAgICByX29yaWVudCAgID0gMi4wICogdyAqIHcgLSAxLjAgICAgICAgICAgICAgICAgICAgICAgICAjIOKIiCBbLTEsICAxXQoKICAgICAgICAjIC0tLSBWZWxvY2l0eSBwZW5hbHR5IChlbmNvdXJhZ2VzIGhvdmVyaW5nLCBub3QgZHJpZnRpbmcpCiAgICAgICAgcl92ZWwgICAgICA9IC0wLjEwICogZmxvYXQobnAuc3VtKHZlbCoqMikpCgogICAgICAgICMgLS0tIEFuZ3VsYXIgdmVsb2NpdHkgcGVuYWx0eQogICAgICAgIHJfYW5ndmVsICAgPSAtMC4wNSAqIGZsb2F0KG5wLnN1bShhbmdfdmVsKioyKSkKCiAgICAgICAgIyAtLS0gQWN0aW9uIHNtb290aG5lc3MgKHBlbmFsaXNlIHN1ZGRlbiB0aHJvdHRsZSBjaGFuZ2VzKQogICAgICAgIHJfc21vb3RoICAgPSAtMC4xMCAqIGZsb2F0KG5wLnN1bSgoYWN0aW9uIC0gc2VsZi5fcHJldl9hY3Rpb24pKioyKSkKCiAgICAgICAgIyAtLS0gQWxpdmUgYm9udXMgKGVuY291cmFnZXMgbG9uZ2VyIGVwaXNvZGVzKQogICAgICAgIHJfYWxpdmUgICAgPSAwLjEwCgogICAgICAgICMgLS0tIENyYXNoIHBlbmFsdHkKICAgICAgICByX2NyYXNoICAgID0gLTEwMC4wIGlmIHBvc1syXSA8IHNlbGYuQ1JBU0hfWiBlbHNlIDAuMAoKICAgICAgICByZXR1cm4gcl9wb3MgKyByX29yaWVudCArIHJfdmVsICsgcl9hbmd2ZWwgKyByX3Ntb290aCArIHJfYWxpdmUgKyByX2NyYXNoCgogICAgIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKICAgICMgR3ltbmFzaXVtIEFQSQogICAgIyDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZDilZAKCiAgICBkZWYgcmVzZXQoCiAgICAgICAgc2VsZiwKICAgICAgICBzZWVkOiBPcHRpb25hbFtpbnRdID0gTm9uZSwKICAgICAgICBvcHRpb25zOiBPcHRpb25hbFtEaWN0XSA9IE5vbmUsCiAgICApOgogICAgICAgIHN1cGVyKCkucmVzZXQoc2VlZD1zZWVkKQogICAgICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQoKICAgICAgICBtdWpvY28ubWpfcmVzZXREYXRhKHNlbGYubW9kZWwsIHNlbGYuZGF0YSkKCiAgICAgICAgIyDilIDilIAgUmFuZG9taXNlZCBpbml0aWFsIHBvc2l0aW9uIChuZWFyIHRhcmdldCkg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgc3ByZWFkICAgICAgID0gbnAuYXJyYXkoWzAuMjAsIDAuMjAsIDAuMTVdKQogICAgICAgIGluaXRfcG9zICAgICA9IHNlbGYudGFyZ2V0X3BvcyArIHJuZy51bmlmb3JtKC1zcHJlYWQsIHNwcmVhZCkKICAgICAgICBpbml0X3Bvc1syXSAgPSBtYXgoaW5pdF9wb3NbMl0sIDAuMTUpICAgICAgICAgICMgbmV2ZXIgc3RhcnQgdW5kZXJncm91bmQKICAgICAgICBzZWxmLmRhdGEucXBvc1swOjNdID0gaW5pdF9wb3MKCiAgICAgICAgIyDilIDilIAgU21hbGwgcmFuZG9tIHRpbHQgKHZhbGlkIHF1YXRlcm5pb24gY2xvc2UgdG8gaWRlbnRpdHkpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHRpbHQgICAgICAgICAgPSBybmcudW5pZm9ybSgtMC4wNSwgMC4wNSwgMykKICAgICAgICBxdWF0ICAgICAgICAgID0gbnAuemVyb3MoNCkKICAgICAgICBxdWF0WzE6NF0gICAgID0gdGlsdAogICAgICAgIHF1YXRbMF0gICAgICAgPSBucC5zcXJ0KG1heCgwLjAsIDEuMCAtIGZsb2F0KG5wLnN1bSh0aWx0KioyKSkpKQogICAgICAgIHF1YXQgICAgICAgICAvPSBucC5saW5hbGcubm9ybShxdWF0KQogICAgICAgIHNlbGYuZGF0YS5xcG9zWzM6N10gPSBxdWF0CgogICAgICAgICMg4pSA4pSAIFNtYWxsIHJhbmRvbSBpbml0aWFsIHZlbG9jaXR5IOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHNlbGYuZGF0YS5xdmVsWzpdID0gcm5nLnVuaWZvcm0oLTAuMDUsIDAuMDUsIHNlbGYubW9kZWwubnYpCgogICAgICAgICMg4pSA4pSAIFJlc2V0IHRyYWNraW5nIHN0YXRlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgICAgIHNlbGYuX3ByZXZfYWN0aW9uWzpdID0gc2VsZi5IT1ZFUl9USFJPVFRMRQogICAgICAgIHNlbGYuX3N0ZXBfY291bnQgICAgICA9IDAKCiAgICAgICAgbXVqb2NvLm1qX2ZvcndhcmQoc2VsZi5tb2RlbCwgc2VsZi5kYXRhKQogICAgICAgIHJldHVybiBzZWxmLl9nZXRfb2JzKCksIHNlbGYuX2dldF9pbmZvKCkKCiAgICBkZWYgc3RlcChzZWxmLCBhY3Rpb246IG5wLm5kYXJyYXkpOgogICAgICAgIGFjdGlvbiA9IG5wLmNsaXAobnAuYXNhcnJheShhY3Rpb24sIGR0eXBlPW5wLmZsb2F0MzIpLCAwLjAsIDEuMCkKCiAgICAgICAgIyBBcHBseSBtb3RvciBjb21tYW5kcyBhbmQgYWR2YW5jZSBzaW11bGF0aW9uCiAgICAgICAgc2VsZi5kYXRhLmN0cmxbOl0gPSBhY3Rpb24KICAgICAgICBmb3IgXyBpbiByYW5nZShzZWxmLk5fU1VCU1RFUFMpOgogICAgICAgICAgICBtdWpvY28ubWpfc3RlcChzZWxmLm1vZGVsLCBzZWxmLmRhdGEpCgogICAgICAgIHNlbGYuX3N0ZXBfY291bnQgKz0gMQoKICAgICAgICByZXdhcmQgICAgICAgICAgICAgICA9IHNlbGYuX2NvbXB1dGVfcmV3YXJkKGFjdGlvbikKICAgICAgICB0ZXJtaW5hdGVkLCB0cnVuY2F0ZWQgPSBzZWxmLl9jaGVja190ZXJtaW5hdGlvbigpCiAgICAgICAgc2VsZi5fcHJldl9hY3Rpb25bOl0gPSBhY3Rpb24KCiAgICAgICAgb2JzICA9IHNlbGYuX2dldF9vYnMoKQogICAgICAgIGluZm8gPSBzZWxmLl9nZXRfaW5mbygpCgogICAgICAgIGlmIHNlbGYucmVuZGVyX21vZGUgPT0gImh1bWFuIjoKICAgICAgICAgICAgc2VsZi5yZW5kZXIoKQoKICAgICAgICByZXR1cm4gb2JzLCByZXdhcmQsIHRlcm1pbmF0ZWQsIHRydW5jYXRlZCwgaW5mbwoKICAgIGRlZiByZW5kZXIoc2VsZik6CiAgICAgICAgaWYgc2VsZi5yZW5kZXJfbW9kZSA9PSAiaHVtYW4iOgogICAgICAgICAgICBpZiBzZWxmLl92aWV3ZXIgaXMgTm9uZToKICAgICAgICAgICAgICAgIHNlbGYuX3ZpZXdlciA9IG11am9jby52aWV3ZXIubGF1bmNoX3Bhc3NpdmUoCiAgICAgICAgICAgICAgICAgICAgc2VsZi5tb2RlbCwgc2VsZi5kYXRhCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuX3ZpZXdlci5zeW5jKCkKCiAgICAgICAgZWxpZiBzZWxmLnJlbmRlcl9tb2RlID09ICJyZ2JfYXJyYXkiOgogICAgICAgICAgICBpZiBzZWxmLl9yZW5kZXJlciBpcyBOb25lOgogICAgICAgICAgICAgICAgc2VsZi5fcmVuZGVyZXIgPSBtdWpvY28uUmVuZGVyZXIoCiAgICAgICAgICAgICAgICAgICAgc2VsZi5tb2RlbCwgaGVpZ2h0PTQ4MCwgd2lkdGg9NjQwCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuX3JlbmRlcmVyLnVwZGF0ZV9zY2VuZShzZWxmLmRhdGEpCiAgICAgICAgICAgIHJldHVybiBzZWxmLl9yZW5kZXJlci5yZW5kZXIoKQoKICAgIGRlZiBjbG9zZShzZWxmKToKICAgICAgICBpZiBzZWxmLl92aWV3ZXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYuX3ZpZXdlci5jbG9zZSgpCiAgICAgICAgICAgIHNlbGYuX3ZpZXdlciA9IE5vbmUKICAgICAgICBpZiBzZWxmLl9yZW5kZXJlciBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5fcmVuZGVyZXIuY2xvc2UoKQogICAgICAgICAgICBzZWxmLl9yZW5kZXJlciA9IE5vbmUK"
_env_src = base64.b64decode(_ENV_B64).decode("utf-8")

# Patch the XML path: the env locates the XML relative to its own __file__.
# In the cloud the env lives at BASE_DIR/envs/drone_env.py, and the XML
# is at BASE_DIR/models/whoop_drone.xml – the relative path "../models/…"
# is already correct, so NO patch is needed.

with open(os.path.join(ENV_DIR, "__init__.py"), "w") as _f:
    _f.write("from .drone_env import WhoopDroneEnv\n"
             "from .drone_env_dr import WhoopDroneEnvDR\n")

with open(os.path.join(ENV_DIR, "drone_env.py"), "w") as _f:
    _f.write(_env_src)

# drone_env_dr.py  ──────────────────────────────────────────────────────
_DR_B64 = "IiIiCldob29wRHJvbmVFbnZEUiDigJMgRG9tYWluIFJhbmRvbWl6YXRpb24gd3JhcHBlcgo9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KU3ViY2xhc3Mgb2YgV2hvb3BEcm9uZUVudi4gIEVhY2ggZXBpc29kZSByZS1yYW5kb21pemVzOgogIOKAoiBNb3RvciB0aHJ1c3QgICAgICDCsTE1ICUgcGVyIG1vdG9yICAobW90b3Igd2VhciAvIHByb3AgaW1iYWxhbmNlKQogIOKAoiBUb3RhbCBtYXNzICAgICAgICDCsTEwICUgICAgICAgICAgICAoYmF0dGVyeSAvIHBheWxvYWQgdmFyaWF0aW9uKQogIOKAoiBDb25zdGFudCB3aW5kICAgICDiiaQwLjggbS9zIGluIGFueSBkaXJlY3Rpb24KICDigKIgT2JzZXJ2YXRpb24gbm9pc2UgR2F1c3NpYW4gz4M9MC4wMSAgKElNVSBub2lzZSkKClVzZSB0aGlzIGZvciBTdGFnZSAyIChzaW0tdG8tcmVhbCBoYXJkZW5pbmcpIGZpbmUtdHVuaW5nIGFmdGVyIHRoZQpiYXNlIHBvbGljeSBoYXMgYWxyZWFkeSBsZWFybmVkIHRvIGhvdmVyIGluIGNsZWFuIHNpbXVsYXRpb24uCiIiIgoKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIE9wdGlvbmFsCgppbXBvcnQgbXVqb2NvCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuZHJvbmVfZW52IGltcG9ydCBXaG9vcERyb25lRW52CgoKY2xhc3MgV2hvb3BEcm9uZUVudkRSKFdob29wRHJvbmVFbnYpOgogICAgIyDilIDilIAgUmFuZG9taXphdGlvbiByYW5nZXMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBEUl9USFJVU1RfUkFOR0UgID0gKDAuODUsIDEuMTUpICAgIyDCsTE1JSBwZXIgbW90b3IKICAgIERSX01BU1NfUkFOR0UgICAgPSAoMC45MCwgMS4xMCkgICAjIMKxMTAlIHRvdGFsIG1hc3MKICAgIERSX1dJTkRfTUFYICAgICAgPSAwLjggICAgICAgICAgICAjIG0vcyBtYXggcGVyIGF4aXMKICAgIERSX09CU19OT0lTRV9TVEQgPSAwLjAxICAgICAgICAgICAjIEdhdXNzaWFuIHN0ZCBhZGRlZCB0byBldmVyeSBvYnMgZGltCgogICAgZGVmIF9faW5pdF9fKHNlbGYsICoqa3dhcmdzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCoqa3dhcmdzKQoKICAgICAgICAjIFN0b3JlIG5vbWluYWwgKFhNTC1kZWZpbmVkKSB2YWx1ZXMgc28gd2UgY2FuIHNjYWxlIGZyb20gdGhlbQogICAgICAgIHNlbGYuX25vbV90aHJ1c3QgPSBzZWxmLm1vZGVsLmFjdHVhdG9yX2dlYXJbOiwgMl0uY29weSgpICAgIyBzaGFwZSAoNCwpCiAgICAgICAgc2VsZi5fbm9tX3RvcnF1ZSA9IHNlbGYubW9kZWwuYWN0dWF0b3JfZ2Vhcls6LCA1XS5jb3B5KCkgICAjIHNoYXBlICg0LCkKICAgICAgICBzZWxmLl9ub21fbWFzcyAgID0gZmxvYXQoc2VsZi5tb2RlbC5ib2R5X21hc3Nbc2VsZi5fZHJvbmVfYm9keV9pZF0pCgogICAgICAgICMgUk5HIGZvciBwZXItc3RlcCBvYnNlcnZhdGlvbiBub2lzZSAoaW5pdGlhbGlzZWQgaGVyZTsgcmVwbGFjZWQgaW4gcmVzZXQpCiAgICAgICAgc2VsZi5fZHJfcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKCkKCiAgICAjIOKUgOKUgCBHeW1uYXNpdW0gQVBJIG92ZXJyaWRlcyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCiAgICBkZWYgcmVzZXQoCiAgICAgICAgc2VsZiwKICAgICAgICBzZWVkOiBPcHRpb25hbFtpbnRdID0gTm9uZSwKICAgICAgICBvcHRpb25zOiBPcHRpb25hbFtEaWN0XSA9IE5vbmUsCiAgICApOgogICAgICAgICMgSW5pdGlhbGlzZSBub2lzZSBSTkcgQkVGT1JFIHN1cGVyKCkucmVzZXQoKSBjYWxscyBfZ2V0X29icygpCiAgICAgICAgc2VsZi5fZHJfcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCgogICAgICAgIF8sIGluZm8gPSBzdXBlcigpLnJlc2V0KHNlZWQ9c2VlZCwgb3B0aW9ucz1vcHRpb25zKQoKICAgICAgICAjIFVzZSBhIHNlcGFyYXRlIFJORyBkZXJpdmVkIGZyb20gc2VlZCBmb3IgRFIgcGFyYW1zIHNvIG5vaXNlIFJORwogICAgICAgICMgYW5kIHBhcmFtZXRlciBSTkcgZG9uJ3Qgc2hhcmUgc3RhdGUuCiAgICAgICAgZHJfcGFyYW1fcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKAogICAgICAgICAgICBOb25lIGlmIHNlZWQgaXMgTm9uZSBlbHNlIHNlZWQgKyAweERFQUQKICAgICAgICApCgogICAgICAgICMg4pSA4pSAIFBlci1tb3RvciB0aHJ1c3QgJiB5YXctdG9ycXVlIHJhbmRvbWlzYXRpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgZm9yIGkgaW4gcmFuZ2UoNCk6CiAgICAgICAgICAgIHNjYWxlID0gZHJfcGFyYW1fcm5nLnVuaWZvcm0oKnNlbGYuRFJfVEhSVVNUX1JBTkdFKQogICAgICAgICAgICBzZWxmLm1vZGVsLmFjdHVhdG9yX2dlYXJbaSwgMl0gPSBzZWxmLl9ub21fdGhydXN0W2ldICogc2NhbGUKICAgICAgICAgICAgc2VsZi5tb2RlbC5hY3R1YXRvcl9nZWFyW2ksIDVdID0gc2VsZi5fbm9tX3RvcnF1ZVtpXSAqIHNjYWxlCgogICAgICAgICMg4pSA4pSAIE1hc3MgcmFuZG9taXNhdGlvbiDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICAgICBtYXNzX3NjYWxlID0gZHJfcGFyYW1fcm5nLnVuaWZvcm0oKnNlbGYuRFJfTUFTU19SQU5HRSkKICAgICAgICBzZWxmLm1vZGVsLmJvZHlfbWFzc1tzZWxmLl9kcm9uZV9ib2R5X2lkXSA9IHNlbGYuX25vbV9tYXNzICogbWFzc19zY2FsZQoKICAgICAgICAjIOKUgOKUgCBXaW5kIHJhbmRvbWlzYXRpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAgICAgc2VsZi5tb2RlbC5vcHQud2luZFs6XSA9IGRyX3BhcmFtX3JuZy51bmlmb3JtKAogICAgICAgICAgICAtc2VsZi5EUl9XSU5EX01BWCwgc2VsZi5EUl9XSU5EX01BWCwgMwogICAgICAgICkKCiAgICAgICAgIyBSZS1mb3J3YXJkIHdpdGggbmV3IHBhcmFtcyAoc3RhdGUgaXMgdW5jaGFuZ2VkOyBmb3JjZXMgYXJlIHVwZGF0ZWQpCiAgICAgICAgbXVqb2NvLm1qX2ZvcndhcmQoc2VsZi5tb2RlbCwgc2VsZi5kYXRhKQoKICAgICAgICAjIFJldHVybiBmcmVzaCBvYnMgKHdpdGggbm9pc2UpIGJ1aWx0IG9uIHRoZSBuZXcgbW9kZWwgc3RhdGUKICAgICAgICByZXR1cm4gc2VsZi5fZ2V0X29icygpLCBpbmZvCgogICAgZGVmIF9nZXRfb2JzKHNlbGYpIC0+IG5wLm5kYXJyYXk6CiAgICAgICAgb2JzICAgPSBzdXBlcigpLl9nZXRfb2JzKCkKICAgICAgICBub2lzZSA9IHNlbGYuX2RyX3JuZy5ub3JtYWwoMC4wLCBzZWxmLkRSX09CU19OT0lTRV9TVEQsIG9icy5zaGFwZSkKICAgICAgICByZXR1cm4gbnAuY2xpcCgKICAgICAgICAgICAgb2JzICsgbm9pc2UuYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICBzZWxmLm9ic2VydmF0aW9uX3NwYWNlLmxvdywKICAgICAgICAgICAgc2VsZi5vYnNlcnZhdGlvbl9zcGFjZS5oaWdoLAogICAgICAgICkK"
with open(os.path.join(ENV_DIR, "drone_env_dr.py"), "wb") as _f:
    _f.write(base64.b64decode(_DR_B64))

print("Environment files written.")


In [ ]:
# ── 5. Imports ──────────────────────────────────────────────────────────────
import os, sys
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    CallbackList, CheckpointCallback, EvalCallback,
)
from stable_baselines3.common.monitor  import Monitor
from stable_baselines3.common.vec_env  import DummyVecEnv, VecNormalize

from envs.drone_env    import WhoopDroneEnv
from envs.drone_env_dr import WhoopDroneEnvDR

print("Imports OK.")
print(f"  WhoopDroneEnv    hover throttle : {WhoopDroneEnv.HOVER_THROTTLE:.3f}")


In [ ]:
# ── 6. Training helper functions ────────────────────────────────────────────

def _env_fn(EnvClass, rank, seed):
    def _init():
        env = EnvClass()
        env = Monitor(env)
        env.reset(seed=seed + rank)
        return env
    return _init


def build_train_env(EnvClass, n_envs, seed, vn_path=None):
    fns = [_env_fn(EnvClass, i, seed) for i in range(n_envs)]
    raw = DummyVecEnv(fns)   # DummyVecEnv avoids fork issues on Colab/Kaggle
    if vn_path and os.path.exists(vn_path):
        vn = VecNormalize.load(vn_path, raw)
        vn.training    = True
        vn.norm_reward = True
    else:
        vn = VecNormalize(raw, norm_obs=True, norm_reward=True,
                          clip_obs=10.0, clip_reward=10.0, gamma=0.99)
    return vn


def build_eval_env(EnvClass, seed, vn_path=None):
    raw = DummyVecEnv([_env_fn(EnvClass, 0, seed)])
    if vn_path and os.path.exists(vn_path):
        vn = VecNormalize.load(vn_path, raw)
        vn.training    = False
        vn.norm_reward = False
    else:
        vn = VecNormalize(raw, norm_obs=True, norm_reward=False,
                          clip_obs=10.0, training=False)
    return vn


def run_stage(cfg, resume_path=None, vn_path=None):
    """Run one training stage; returns (model_zip_path, vn_pkl_path)."""
    run_name = cfg["run_name"]
    n_envs   = cfg["n_envs"]
    seed     = cfg["seed"]
    save_dir = cfg["save_dir"]
    log_dir  = cfg["log_dir"]

    os.makedirs(os.path.join(save_dir, "best"), exist_ok=True)
    os.makedirs(os.path.join(log_dir,  run_name), exist_ok=True)

    train_env = build_train_env(cfg["EnvClass"], n_envs, seed, vn_path)
    eval_env  = build_eval_env (cfg["EnvClass"], seed + 9999, vn_path)

    ckpt_cb = CheckpointCallback(
        save_freq  = max(1, cfg["checkpoint_freq"] // n_envs),
        save_path  = save_dir,
        name_prefix= run_name,
        save_vecnormalize=True,
        verbose=1,
    )
    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path = os.path.join(save_dir, "best"),
        log_path             = os.path.join(log_dir, run_name),
        eval_freq            = max(1, cfg["eval_freq"] // n_envs),
        n_eval_episodes      = 10,
        deterministic        = True,
        render               = False,
        verbose              = 1,
    )

    policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

    if resume_path and os.path.exists(resume_path):
        print(f"[{run_name}] Resuming from {resume_path}")
        model = PPO.load(
            resume_path, env=train_env,
            learning_rate = cfg["learning_rate"],
            clip_range    = cfg["clip_range"],
            ent_coef      = cfg["ent_coef"],
            n_epochs      = cfg["n_epochs"],
            tensorboard_log = log_dir, verbose=1,
        )
    else:
        model = PPO(
            "MlpPolicy", train_env,
            learning_rate  = cfg["learning_rate"],
            n_steps        = cfg["n_steps"],
            batch_size     = cfg["batch_size"],
            n_epochs       = cfg["n_epochs"],
            gamma=0.99, gae_lambda=0.95,
            clip_range     = cfg["clip_range"],
            ent_coef       = cfg["ent_coef"],
            vf_coef=0.5, max_grad_norm=0.5,
            policy_kwargs  = policy_kwargs,
            tensorboard_log= log_dir,
            verbose=1, seed=seed,
        )

    print(f"\n{'='*55}")
    print(f"  {run_name.upper()}")
    print(f"  Env    : {cfg['EnvClass'].__name__}")
    print(f"  n_envs : {n_envs}    steps : {cfg['total_timesteps']:,}")
    print(f"{'='*55}\n")

    model.learn(
        total_timesteps     = cfg["total_timesteps"],
        callback            = CallbackList([ckpt_cb, eval_cb]),
        tb_log_name         = run_name,
        progress_bar        = True,
        reset_num_timesteps = (resume_path is None),
    )

    out_zip = os.path.join(save_dir, f"{run_name}_final")
    out_vn  = os.path.join(save_dir, f"vec_normalize_{run_name}.pkl")
    model.save(out_zip)
    train_env.save(out_vn)

    print(f"[{run_name}] Saved → {out_zip}.zip")
    train_env.close()
    eval_env.close()
    return f"{out_zip}.zip", out_vn


print("Helper functions defined.")


In [ ]:
# ── 7a. Stage 1 configuration ───────────────────────────────────────────────
# Adjust n_envs to the number of CPU cores available on the cloud instance.
# Colab free tier : 2 vCPU  → n_envs = 2
# Colab Pro/Pro+  : 8 vCPU  → n_envs = 8
# Kaggle          : 4 vCPU  → n_envs = 4

STAGE1_CFG = dict(
    run_name        = "stage1_base",
    EnvClass        = WhoopDroneEnv,
    total_timesteps = 5_000_000,
    n_envs          = 2,           # ← change to match your vCPU count
    seed            = 42,
    learning_rate   = 3e-4,
    n_steps         = 2048,
    batch_size      = 64,
    n_epochs        = 10,
    clip_range      = 0.2,
    ent_coef        = 0.001,
    log_dir         = os.path.join(BASE_DIR, "logs"),
    save_dir        = os.path.join(BASE_DIR, "models", "trained"),
    checkpoint_freq = 100_000,
    eval_freq       = 50_000,
)
print("Stage 1 config ready.")


In [ ]:
# ── 7b. Stage 1 – Run base training ────────────────────────────────────────
s1_zip, s1_vn = run_stage(STAGE1_CFG)
print(f"\nStage 1 complete.\n  model : {s1_zip}\n  vn    : {s1_vn}")


In [ ]:
# ── 8a. Stage 2 configuration (Domain Randomization) ───────────────────────
STAGE2_CFG = dict(
    run_name        = "stage2_dr",
    EnvClass        = WhoopDroneEnvDR,
    total_timesteps = 2_000_000,
    n_envs          = 2,           # ← same as Stage 1
    seed            = 123,
    learning_rate   = 1e-4,        # lower LR for fine-tuning
    n_steps         = 2048,
    batch_size      = 64,
    n_epochs        = 5,           # fewer epochs → less catastrophic forgetting
    clip_range      = 0.15,        # tighter clip → conservative updates
    ent_coef        = 0.0005,
    log_dir         = os.path.join(BASE_DIR, "logs"),
    save_dir        = os.path.join(BASE_DIR, "models", "trained"),
    checkpoint_freq = 50_000,
    eval_freq       = 25_000,
)
print("Stage 2 config ready.")


In [ ]:
# ── 8b. Stage 2 – Domain-randomization fine-tune ───────────────────────────
s2_zip, s2_vn = run_stage(STAGE2_CFG, resume_path=s1_zip, vn_path=s1_vn)
print(f"\nStage 2 complete.\n  model : {s2_zip}\n  vn    : {s2_vn}")


In [ ]:
# ── 9. Headless evaluation ──────────────────────────────────────────────────
N_EVAL_EPS = 10

_eval_env = build_eval_env(WhoopDroneEnvDR, seed=777, vn_path=s2_vn)
_model    = PPO.load(s2_zip, device="cpu")

ep_rewards, ep_lengths, ep_dists = [], [], []

for _ep in range(N_EVAL_EPS):
    _obs  = _eval_env.reset()
    _done = False
    _r, _s, _d = 0.0, 0, []

    while not _done:
        _act, _ = _model.predict(_obs, deterministic=True)
        _obs, _rew, _done_arr, _info_arr = _eval_env.step(_act)
        _r   += float(_rew[0])
        _s   += 1
        _done = bool(_done_arr[0])
        if "distance_to_target" in _info_arr[0]:
            _d.append(_info_arr[0]["distance_to_target"])

    ep_rewards.append(_r)
    ep_lengths.append(_s)
    ep_dists.append(float(np.mean(_d)) if _d else float("nan"))
    print(f"  Ep {_ep+1:2d}:  reward={_r:8.1f}  steps={_s:5d}  "
          f"dist={ep_dists[-1]:.3f} m")

print("─" * 50)
print(f"  Mean reward : {np.mean(ep_rewards):.2f} ± {np.std(ep_rewards):.2f}")
print(f"  Mean steps  : {np.mean(ep_lengths):.0f}")
print(f"  Mean dist   : {np.nanmean(ep_dists):.3f} m")

_eval_env.close()


In [ ]:
# ── 10. Package artefacts and download ──────────────────────────────────────
import zipfile, shutil

_archive_name = os.path.join(BASE_DIR, "whoop_drone_trained.zip")

with zipfile.ZipFile(_archive_name, "w", zipfile.ZIP_DEFLATED) as _zf:
    for _fpath in [s1_zip, s1_vn, s2_zip, s2_vn]:
        if _fpath and os.path.exists(_fpath):
            _zf.write(_fpath, os.path.basename(_fpath))
    # Also include best model if present
    _best = os.path.join(BASE_DIR, "models", "trained", "best", "best_model.zip")
    if os.path.exists(_best):
        _zf.write(_best, "best_model.zip")
    # Include best VecNorm if present
    _best_vn = os.path.join(BASE_DIR, "models", "trained", "best", "vec_normalize.pkl")
    if os.path.exists(_best_vn):
        _zf.write(_best_vn, "best_vec_normalize.pkl")

print(f"Archive created: {_archive_name}")
print(f"Size: {os.path.getsize(_archive_name) / 1024 / 1024:.1f} MB")

# ── Download ─────────────────────────────────────────────────────────────
try:
    from google.colab import files
    files.download(_archive_name)
    print("Download started (Colab).")
except ImportError:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        print(f"Kaggle: find the archive in the Output tab: {_archive_name}")
    else:
        print(f"File saved locally at: {_archive_name}")
